# 01 — Structure mapping and contacts (NUDT15 / 5lpg)

Load the shipped 5lpg chain A UniProt map, check the SEQRES→PDB offset, then
compute ligand and protein contacts from the **raw** 5lpg PDB (ligands are
stripped from the cleaned file).

Outputs: `examples/NUDT15_HUMAN/outputs/5lpg_contacts.csv` and
`5lpg_functional_residues.csv`.


## Paths, sequence, mapping JSON


In [ ]:
from IPython.display import display
from dualde.contacts import (
    NUDT15_LIGAND_NAMES,
    functional_residues,
    residue_contacts,
)
from dualde.mapping import (
    load_structural_mapping_json,
    read_fasta,
    select_mapping_entry,
    seqres_to_pdb_offset,
)
from dualde.paths import example_dir

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

header, uniprot_seq = read_fasta(EXAMPLE / "NUDT15_HUMAN.fasta")
print(header, len(uniprot_seq), "aa")
print(uniprot_seq[:60], "...")


In [ ]:
entries = load_structural_mapping_json(EXAMPLE / "mapping" / "5lpg_A.json")
entry = select_mapping_entry(entries, pdb_id="5lpg", chain="A", uniprot_id="NUD15_HUMAN")

print("keys:", sorted(entry.keys()))
print("pdb / chain / UniProt:", entry["pdb_id"], entry["pdb_chain"], entry["uniprot_ac"], entry["uniprot_id"])
print("UniProt span:", entry["uniprot_start"], "–", entry["uniprot_end"])
print("position_mapping:")
for seg in entry["position_mapping"]:
    print(" ", seg)


## SEQRES offset and contacts


In [ ]:
raw_pdb = EXAMPLE / "pdb" / "5lpg.pdb"
offset = seqres_to_pdb_offset(str(raw_pdb), "5lpg", "A", search_dirs=[str(EXAMPLE / "pdb")])
print("SEQRES→PDB offset (expect 0 for 5lpg):", offset)

contacts = residue_contacts(
    str(raw_pdb),
    entry,
    chain="A",
    ligand_names=NUDT15_LIGAND_NAMES,
    distance_cutoff=8.0,
    seqres_to_pdb_offset=offset,
    include_protein=True,
)
contacts.to_csv(OUT / "5lpg_contacts.csv", index=False)
print("wrote", OUT / "5lpg_contacts.csv")


Inspect the long contact table (one row per protein residue × partner within 8 Å). 5lpg's ligand in the named set is **71V**.


In [ ]:
print("shape:", contacts.shape)
print("columns:", list(contacts.columns))
print(contacts["contact_type"].value_counts())
print("ligand names present:", sorted(contacts.loc[contacts["contact_type"] == "ligand", "residue_name"].unique()))
contacts.head()


In [ ]:
func = functional_residues(contacts, distance_cutoff=3.5)
func.to_csv(OUT / "5lpg_functional_residues.csv", index=False)

print("functional residues (< 3.5 Å):", len(func))
print(func["contact_type"].value_counts())
print("ligand sites:")
display(func.loc[func["contact_type"] == "ligand", ["target_pos", "residue_name", "minimum_distance"]])
print("closest protein contacts (first 8):")
func.loc[func["contact_type"] == "protein"].head(8)


To rebuild the mapping from SIFTS instead of the shipped JSON, download
[pdb_chain_uniprot.csv.gz](https://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/csv/pdb_chain_uniprot.csv.gz)
and call `dualde.mapping.download_sifts_table` + `mapping_entry_from_sifts`
(5lpg, chain A, UniProt Q9NV35).
